In [ ]:
import sys
from pathlib import Path
root = Path().resolve().parent.parent   
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from utils.grade_submit import submit_grade
# Initialize Otter
import otter
grader = otter.Notebook("hw03.ipynb")

<img src="data8logo.png" alt="Data 8 Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Домашнє завдання 3: Маніпуляція таблицями та візуалізація

Будь ласка, заповніть цей записник, заповнюючи надані комірки. Перед початком виконайте попередню комірку, щоб завантажити надані тести.

**Корисний ресурс:**
- [Python Reference](http://data8.org/sp26/reference/): Шпаргалка з корисними методами для масивів (array) і таблиць (table), що використовуються в Data 8!

**Рекомендоване читання**: 
* [Visualization](https://inferentialthinking.com/chapters/07/Visualization.html)

Для всіх задач, у яких ви повинні писати пояснення та речення, ви **повинні** надати свою відповідь у відведеному місці. Крім того, протягом цього домашнього завдання та всіх наступних **будь ласка, не переприсвоюйте змінні протягом усього записника!** Наприклад, якщо ви використовуєте `max_temperature` у відповіді на одне питання, не переприсвоюйте її пізніше. Інакше ви не пройдете тести, які, як вам здавалося, ви вже проходили раніше!

**Дедлайн:**

Це завдання **має бути здане в середу, 11.02 о 11:00 ранку за тихоокеанським часом (PT)**. Роботи, подані після цього часу, будуть прийняті протягом 24 годин і отримають штраф у 20%. Будь-які роботи, подані пізніше цього 24-годинного періоду, не будуть прийняті, якщо не буде надано продовження терміну згідно зі [сторінкою силабусу](http://data8.org/sp26/syllabus/). Здайте роботу до вівторка, 10.02 о 11:00 ранку PT, щоб отримати 5 додаткових балів.

Безпосередній обмін відповідями не дозволяється, але обговорення задач з викладачами курсу або іншими студентами заохочується. Зверніться до [сторінки силабусу](https://www.data8.org/sp26/syllabus), щоб дізнатися більше про те, як навчатися спільно.

Вам слід почати рано, щоб мати час отримати допомогу, якщо ви застрягнете. Години консультацій проводяться з понеділка по п'ятницю у [Warren Hall](https://www.berkeley.edu/map/warren-hall/) та онлайн. Розклад годин консультацій розміщено на нашій [сторінці годин консультацій](https://www.data8.org/sp26/calendar/).

---

Розподіл балів за це завдання наведено в таблиці нижче:
| Категорія | Бали |
| --- | --- |
| Автоперевірка (питання з кодуванням) | 87 |
| Ручна перевірка (питання з візуалізації) | 13 |
| **Всього** | 100 |


In [ ]:
# Don't change this cell; just run it. 

import numpy as np
from datascience import *
import warnings
warnings.simplefilter('ignore', FutureWarning)

# These lines do some fancy plotting magic.\n",
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1. Рівні народжуваності

Наступна таблиця містить оцінки чисельності населення на основі даних Бюро перепису населення США для кожного штату США станом на 1 липня 2015 року та 1 липня 2016 року. Останні чотири стовпці описують складові оціненої зміни чисельності населення протягом цього часового інтервалу. **Для всіх питань нижче припускайте, що слово «штати» стосується всіх 52 рядків, включаючи Пуерто-Рико та Округ Колумбія.**

Дані взяті з [тут](http://www2.census.gov/programs-surveys/popest/datasets/2010-2016/national/totals/nst-est2016-alldata.csv). (Примітка: Якщо файл не завантажується при натисканні на посилання, ви можете скопіювати та вставити адресу посилання в адресний рядок!) Якщо ви хочете дізнатися більше про описи різних стовпців, натисніть [тут](http://www2.census.gov/programs-surveys/popest/datasets/2010-2015/national/totals/nst-est2015-alldata.pdf).

Сирі дані трохи заплутані — виконайте комірку нижче, щоб очистити таблицю та зробити її зручнішою для роботи.

In [ ]:
# Don't change this cell; just run it.
pop = Table.read_table('nst-est2016-alldata.csv').where('SUMLEV', 40).select([1, 4, 12, 13, 27, 34, 62, 69])
pop = pop.relabeled('POPESTIMATE2015', 'START_POP').relabeled('POPESTIMATE2016', 'END_POP')
pop = pop.relabeled('BIRTHS2016', 'BIRTHS').relabeled('DEATHS2016', 'DEATHS')
pop = pop.relabeled('NETMIG2016', 'MIGRATION').relabeled('RESIDUAL2016', 'OTHER')
pop = pop.with_columns("REGION", np.array([int(region) if region != "X" else 0 for region in pop.column("REGION")]))
pop.set_format([2, 3, 4, 5, 6, 7], NumberFormatter(decimals=0)).show(5)

в таблиці яку ви побачите після виконання попередньої комірки, стовпчики містять таку інформацію:
- `REGION`- код регіона до якого належить штат
- `NAME` - назва штату	
- `START_POP` - розмір популяції людей в кожному штаті США станом на 1 липня 2015 року (початок періода спостереження)
- `END_POP`	- розмір популяції людей в кожному штаті США станом на 1 липня 2016 року (через рік)	
- `BIRTHS` - скільки за рік народилось людей в кожному штаті
- `DEATHS` - скільки за рік померло людей в кожному штаті	
- `MIGRATION` - скільки за рік мігрувало людей в кожному штаті	
- `OTHER` - скільки за рік людей не стало або збільшилось в кожному штаті з інших причин

---

**Питання 1.1.** Присвойте `us_birth_rate` загальний річний коефіцієнт народжуваності США протягом цього часового інтервалу. Річний коефіцієнт народжуваності для річного періоду — це загальна кількість народжень у цьому періоді як частка (proportion) від загального розміру населення на початку часового періоду. **(7 балів)**

_Підказка:_ Що представляє кожен рядок у таблиці `pop`? Як ми можемо використати це, щоб знайти загальне населення США?


In [ ]:
us_birth_rate = ...
us_birth_rate

In [ ]:
grader.check("q1_1")

---

**Питання 1.2.** Присвойте `movers` кількість штатів, для яких **абсолютне значення** **річного коефіцієнта міграції** було вищим за 1%. Річний коефіцієнт міграції для річного періоду — це чиста кількість міграцій (в'їзд мінус виїзд) як частка (proportion) від розміру населення на початку періоду. Стовпець `MIGRATION` містить оцінені річні чисті кількості міграцій за штатами. **(7 балів)**

*Підказка*: `migration_rates` має бути таблицею (table), а `movers` — числом.

*Підказка*: в своїй відповіді ви можете використати такі методи та атрибути таблиць: `column`, `with_column`, `where`, `num_rows`. Також функцію `abs` бібіліотеки `numpy`.


In [ ]:
migration_rates = ...
movers = ...
movers

In [ ]:
grader.check("q1_2")

---

**Питання 1.3.** Присвойте `west_births` загальну кількість народжень, які відбулися в регіоні 4 (Захід США). **(7 балів)**

*Підказка:* Обов'язково двічі перевірте тип значень у стовпці `REGION` і відповідно відфільтруйте (тобто типи мають збігатися!).


In [ ]:
west_births = ...
west_births

In [ ]:
grader.check("q1_3")

---

**Питання 1.4.** У наступному питанні ви створюватимете візуалізацію, щоб зрозуміти зв'язок між коефіцієнтами народжуваності та смертності. Річний коефіцієнт смертності для річного періоду — це загальна кількість смертей у цьому періоді як частка (proportion) від розміру населення на початку часового періоду.

Яка візуалізація є найбільш підходящою, щоб побачити, чи існує асоціація між річними коефіцієнтами народжуваності та смертності в різних штатах Сполучених Штатів?

1. Лінійний графік (Line Graph)
2. Стовпчаста діаграма (Bar Chart)
3. Діаграма розсіювання (Scatter Plot)

Присвойте змінній `visualization` в комірці нижче число, що відповідає правильному варіанту візуалізації. **(6 балів)**


In [ ]:
visualization = ...

In [ ]:
grader.check("q1_4")

<!-- BEGIN QUESTION -->

--- 

**Питання 1.5.** У комірці з кодом нижче створіть візуалізацію, яка допоможе нам визначити, чи існує асоціація між коефіцієнтом народжуваності та коефіцієнтом смертності протягом цього часового інтервалу. Може бути корисним створити проміжну таблицю, що містить коефіцієнти народжуваності та смертності для кожного штату. **(5 балів)**

Речі, які варто врахувати:

- Який тип діаграми допоможе нам проілюструвати асоціацію між 2 змінними?
- Як ви можете маніпулювати певною таблицею, щоб допомогти згенерувати вашу діаграму?
- Перегляньте [Рекомендоване читання](https://inferentialthinking.com/chapters/07/Visualization.html) для цього домашнього завдання!

**Примітка: Усі питання з побудови графіків і візуалізації в цьому домашньому завданні будуть перевірені вручну на правильність. Навіть якщо ці питання не мають комірки перевірки grader, вони все одно враховуватимуться у вашій оцінці!**

In [ ]:
# In this cell, use birth_rates and death_rates to generate your visualization
birth_rates = pop.column('BIRTHS') / pop.column('START_POP')
death_rates = pop.column('DEATHS') / pop.column('START_POP')
...

<!-- END QUESTION -->

---

**Питання 1.6.** Правда чи неправда: Існує асоціація між коефіцієнтом народжуваності та коефіцієнтом смертності протягом цього часового інтервалу. 

Присвойте `assoc` значення `True` або `False` у комірці нижче. **(7 балів)**


In [ ]:
assoc = ...

In [ ]:
grader.check("q1_6")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 2. Uber

**Примітка:** Ми рекомендуємо прочитати [Розділ 7.2](https://inferentialthinking.com/chapters/07/2/Visualizing_Numerical_Distributions.html) підручника перед початком Питання 3.

Нижче ми завантажуємо таблиці, що містять 200 000 поїздок Uber у будні дні в метрополіях Маніли (Філіппіни) та Бостона (Массачусетс) з проєкту [Uber Movement](https://www.uber.com/newsroom/introducing-uber-movement-2/). Стовпці `sourceid` і `dstid` містять коди, що відповідають початковим і кінцевим локаціям кожної поїздки. Стовпець `hod` містить коди, що відповідають годині дня, коли відбулася поїздка. Стовпець `ride time` містить тривалість поїздки в хвилинах.

In [ ]:
boston = Table.read_table("boston.csv")
manila = Table.read_table("manila.csv")
print("Boston Table")
boston.show(4)
print("Manila Table")
manila.show(4)

<!-- BEGIN QUESTION -->

--- 

**Питання 2.1.** Побудуйте гістограму, яка візуалізує розподіли всіх тривалостей поїздок у Бостоні, використовуючи задані біни (bins) у `equal_bins`. **(4 бали)**

*Підказка:* Див. [Розділ 7.2](https://inferentialthinking.com/chapters/07/2/Visualizing_Numerical_Distributions.html), якщо ви застрягли з тим, як задати біни (bins).

**Примітка: Усі питання з побудови графіків і візуалізації в цьому домашньому завданні будуть перевірені вручну на правильність. Навіть якщо ці питання не мають комірки перевірки grader, вони все одно враховуватимуться у вашій оцінці!**

In [ ]:
equal_bins = np.arange(0, 120, 5)
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

**Питання 2.2.** Тепер побудуйте гістограму, яка візуалізує розподіл усіх тривалостей поїздок у Манілі, використовуючи задані біни (bins). **(4 бали)**


In [ ]:
equal_bins = np.arange(0, 120, 5)
...

# Don't delete the following line!
plt.ylim(0, 0.05);

<!-- END QUESTION -->

---

**Питання 2.3.** Давайте детальніше розглянемо підпис (мітку, label) осі y гістограм які ви будували. Присвойте змінній `unit_meaning` в наступній комірці, ціле число (1, 2, 3), яке відповідає слову «одиниця» в фразі «Відсоток на одиницю» (Percent per unit) яка є підписом вісі. **(7 балів)**

1. хвилина (minute)  
2. тривалість поїздки (ride time)  
3. секунда (second)


In [ ]:
unit_meaning = ...
unit_meaning

In [ ]:
grader.check("q2_3")

---

**Питання 2.4.** Присвойте `boston_under_15` і `manila_under_15` відсоток поїздок, які тривають менше 15 хвилин у відповідних метрополіях. Використовуйте змінні висоти, надані нижче, щоб обчислити відсотки. Ваше рішення має використовувати лише змінні висоти, числа та математичні операції. Ви **не** повинні звертатися до таблиць `boston` і `manila` жодним чином. **(8 балів)**

> ***Примітка:*** змінні висоти (тобто `boston_under_5`) представляють висоту біна (bin), який вони описують.


In [ ]:
boston_under_5_bin_height = 1.2
manila_under_5_bin_height = 0.6
boston_5_to_under_10_bin_height = 3.2
manila_5_to_under_10_bin_height = 1.4
boston_10_to_under_15_bin_height = 4.9
manila_10_to_under_15_bin_height = 2.2

boston_under_15 = ...
manila_under_15 = ...

boston_under_15, manila_under_15

In [ ]:
grader.check("q2_4")

---

**Питання 2.5.** Давайте детальніше розглянемо розподіл тривалостей поїздок у Бостоні. Присвойте `boston_median_bin` ціле число (1, 2, 3 або 4), яке відповідає біну (bin), що містить медіану часу. **(6 балів)**

1. 0-8 хвилин  
2. 8-14 хвилин  
3. 14-20 хвилин  
4. 20-40 хвилин  

*Підказка:* Медіана відсортованого списку має половину елементів списку ліворуч від неї та половину праворуч.
Якщо ви маєте список чисел (наприклад 10,3,11,1,5) знайди для них медіану можна наступним чином: 1) відсортуйте список в порядку збільшення (1,3,5,10,11). 2) Знайдіть число яке стоїть в середині відсортованого списку - очевидно, що половина чисел буде ліворуч від нього, друга половина праворуч - це число і є медіана (5). В вашому випадку (значення часу на вісі абсцис) числа вже відсортовані.


In [ ]:
boston_median_bin = ...
boston_median_bin

In [ ]:
grader.check("q2_5")

---

**Питання 2.6.** Визначте основні відмінності між двома гістограмами з точки зору статистичних властивостей.

Присвойте всі правильні твердження масиву (array) `histogram_answer` (наприклад, `histogram_answer = make_array(1,2,3)`).
> *Підказка*: Не виконуючи жодних обчислень, що ви можете спостерігати щодо середнього значення або скошеності кожної гістограми? Розподіл має праву скошеність (right skew), якщо він має довгий правий «хвіст» значень. **(4 бали)**

1. У Маніли був вищий середній час поїздки, ніж у Бостоні.
2. У Бостона був вищий середній час поїздки, ніж у Маніли.
3. Гістограма Маніли була більш скошеною вправо, ніж гістограма Бостона.
4. Гістограма Бостона була більш скошеною вправо, ніж гістограма Маніли.


In [ ]:
histogram_answer = ...

In [ ]:
grader.check("q2_6")

--- 

**Питання 2.7.** Бостон відомий дуже холодними зимами, тоді як Маніла відома важким трафіком. Ці фактори можуть допомогти пояснити відмінності, які ви спостерігали в Питанні 2.6. 

Присвойте всі твердження, які б підтримували наші спостереження щодо тривалостей поїздок у Бостоні та Манілі, масиву (array) `difference_factors`. **(4 бали)**

1. Холодніша погода в Бостоні спонукає людей замінювати ходьбу короткими поїздками, що призводить до коротших середніх тривалостей поїздок.
2. Холодніша погода в Бостоні призводить до затримок, пов'язаних із погодою, що призводить до довших середніх тривалостей поїздок.
3. Важкий трафік у Манілі спонукає людей уникати подорожей на великі відстані, що призводить до коротших середніх тривалостей поїздок.
4. Важкий трафік у Манілі спричиняє часті затримки, що призводить до довших середніх тривалостей поїздок.


In [ ]:
difference_factors = ...

In [ ]:
grader.check("q2_7")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. Гістограми

Розглянемо наступну діаграму розсіювання: 

![Alt text](scatter.png "Scatter plot showing data points for the variables 'x' and 'y'. The data are symmetric about the x-axis centered at 0 and symmetric about the y-axis centered at 0, but with no data in the [-0.5, 0.5] range on the y-axis.")

Осі графіка представляють значення двох змінних: $x$ та $y$. 

Припустимо, у нас є таблиця під назвою `t`, яка має два стовпці:

- `x`: стовпець, що містить x-значення точок на діаграмі розсіювання
- `y`: стовпець, що містить y-значення точок на діаграмі розсіювання

Нижче вам надано три гістограми — одна відповідає стовпцю `x`, одна відповідає стовпцю `y`, а одна не відповідає жодному зі стовпців. 

**Гістограма A:**
 
![Alt text](var3.png "Symmetrical, bell-shaped histogram centered around 0")

**Гістограма B:**

![Alt text](var1.png "Symmetrical histogram with two peaks at -1 and 1 but no data around 0")

**Гістограма C:**

![Alt text](var2.png "Asymmetrical histogram with a peak around -0.5 and a right skew")

---

**Питання 3.1.** Припустимо, ми виконуємо `t.hist('x')`. Яку гістограму створює цей код? Присвойте `histogram_column_x` одне з чисел 1, 2 або 3. **(7 балів)**

1. Гістограма A
2. Гістограма B
3. Гістограма C


In [ ]:
histogram_column_x = ...

In [ ]:
grader.check("q3_1")

---

**Питання 3.2.** Виберіть усі дійсні причини, чому ви обрали гістограму з Питання 1, і присвойте свої відповіді масиву (array) `histogram_reasoning_x`. **(5 балів)**

1. Оскільки діаграма розсіювання показує два приблизно симетричні кластери ліворуч і праворуч від 0 (на осі $x$), гістограма для $x$ має бути приблизно симетричною навколо 0.
2. Оскільки немає значних прогалин у значеннях змінної $x$, ми очікували б, що гістограма для $x$ не матиме прогалин.
3. Оскільки діаграма розсіювання показує два окремі кластери, гістограма $x$ має мати два піки з порожньою прогалиною між ними.
4. Значення змінної $x$ коливаються приблизно від -2 до 3, тому гістограма також повинна мати діапазон значень приблизно від -2 до 3.
5. Оскільки два кластери на діаграмі розсіювання перекриваються в області між -1 і 0, ми очікували б, що біни (bins) будуть вищими навколо області від -1 до 0 гістограми, оскільки кожен вертикальний зріз у цьому діапазоні містить більше точок.


In [ ]:
histogram_reasoning_x = ...

In [ ]:
grader.check("q3_2")

---

**Питання 3.3.** Припустимо, ми виконуємо `t.hist('y')`. Яку гістограму створює цей код? Присвойте `histogram_column_y` одне з чисел 1, 2 або 3. **(7 балів)**

1. Гістограма A
2. Гістограма B
3. Гістограма C


In [ ]:
histogram_column_y = ...

In [ ]:
grader.check("q3_3")

---

**Питання 3.4.** Виберіть усі дійсні причини, чому ви обрали гістограму з Питання 3, і присвойте свої відповіді масиву (array) `histogram_reasoning_y` **(5 балів)**

1. Оскільки діаграма розсіювання показує два приблизно симетричні кластери ліворуч і праворуч від 0 (на осі $y$), гістограма для $y$ має бути приблизно симетричною навколо 0.
2. Значення змінної $y$ коливаються приблизно від -1.5 до 1.5, тому гістограма також повинна мати діапазон значень приблизно від -1.5 до 1.5.
3. Оскільки діаграма розсіювання показує два окремі кластери, гістограма $y$ має мати два піки з порожньою прогалиною між ними.
4. У точках у напрямку $y$ немає прогалин, тому ми не очікували б прогалини в гістограмі цих значень.
5. Оскільки дві маси на діаграмі розсіювання перекриваються в області між -1 і 0, ми очікували б, що біни (bins) будуть вищими навколо області від -1 до 0 гістограми, оскільки кожен вертикальний зріз у цьому діапазоні містить більше точок.


In [ ]:
histogram_reasoning_y = ...

In [ ]:
grader.check("q3_4")


---

Ви завершили лабораторну роботу!

**Важлива інформація щодо подання роботи:**
- **Виконайте всі тести** і переконайтеся, що всі вони виконані успішно
- **Збережіть** роботу через меню **«Файл»**

**Ви несете відповідальність за те, щоб переконатися, що ваша робота збережена, перш ніж запускати останню комірку.**


## Улюбленці Data 8

Еш, Скаут і Жюль вважають, що ви заслуговуєте на спокійний сон після завершення цього завдання!

<img src="ash.png" width="35%" alt="sleepy cat"/>
<img src="scout.jpg" width="12%" alt="black lab"/>
<img src="jules.jpg" width="19.5%" alt="chaotic bunny"/>

Вітаємо із завершенням Домашнього завдання 3!

---

Щоб двічі перевірити свою роботу, комірка нижче повторно запустить усі тести автоперевірки.

In [ ]:
grader.check_all()

## Подання

Перед виконанням наведеної нижче комірки переконайтеся, що ви виконали всі комірки у своєму зошиті послідовно, щоб у результаті відобразилися всі зображення та графіки. Після виконання двох наведених нижче коміркок ваші результати буде передано викладачу.

#### 1) Напишіть своє ім'я та призвище
В наступній комірці впишіть свої власні ім'я та призвище.  
Запустіть комірку

In [ ]:
# змінній student_name присвойте значення вашого імені та прізвища, як у прикладі нижче.
# student_name = "Андрій Ковальчук"
student_name = 

#### 2) Наступну комірку просто запустіть

In [ ]:
lab_name = "hw03"
comment = "hist"
r = grader.check_all()
submit_grade(student_name, lab_name, comment, grader.check_all(), True)